<a href="https://colab.research.google.com/github/Rumas0/Thesis_work_SSL-imbalance/blob/main/SSL_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**SSL Finetuning**

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import json
from google.colab import drive

In [17]:
drive.mount('/content/drive')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#**Loading Labeled data**

In [19]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os
import pandas as pd


backup_dir = '/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1'
files = ['expA_train.csv', 'expA_val.csv', 'expA_test.csv']

for f in files:
    shutil.copy(f'{backup_dir}/{f}', '.')
    print(f'Loaded {f}')

print(f'\nExpereiment A Training: {len(pd.read_csv("expA_train.csv"))} images')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded expA_train.csv
Loaded expA_val.csv
Loaded expA_test.csv

Expereiment A Training: 483 images


In [22]:
train_df = pd.read_csv('/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1/expA_train.csv')
val_df = pd.read_csv('/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1/expA_val.csv')
test_df = pd.read_csv('/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1/expA_test.csv')

class SkinDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f'data/labeled/{row["image"]}.jpg').convert('RGB')
        label = self.class_to_idx[row['label']]
        if self.transform:
            img = self.transform(img)
        return img, label

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = SkinDataset(train_df, transform)
val_ds = SkinDataset(val_df, transform)
test_ds = SkinDataset(test_df, transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

print(f"Classes: {train_ds.classes}")
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Classes: ['BKL', 'MEL', 'NV']
Train: 483, Val: 104, Test: 104


#**SSL Model Build (Encoder+Classifier)**

In [23]:
class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
    def forward(self, x):
        return self.features(x)

class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

#**Pretrained Weights Loaded**

In [26]:
# Loading pretrained encoder
encoder = SimpleEncoder()
encoder.load_state_dict(torch.load('/content/drive/MyDrive/thesis_backup_day1/ssl_encoder_pretrained.pth'))
print("Loaded SSL pretrained encoder")

Loaded SSL pretrained encoder


In [27]:
# Freezing encoder for stability
for param in encoder.parameters():
    param.requires_grad = False

model = SSLClassifier(encoder, len(train_ds.classes)).to(device)

#**Image Generation**

In [30]:
import os
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm

# Load your CSVs
train_df = pd.read_csv('/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1/expA_train.csv')
val_df = pd.read_csv('/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1/expA_val.csv')
test_df = pd.read_csv('/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1/expA_test.csv')

# Get all unique image IDs
all_images = pd.concat([train_df, val_df, test_df])['image'].unique()

# Recreate synthetic images
def create_synthetic_skin_image(path, size=224):
    base = np.random.randint(100, 200, (size, size, 3), dtype=np.uint8)
    x, y = np.meshgrid(np.arange(size), np.arange(size))
    center_x, center_y = np.random.randint(size//3, 2*size//3, 2)
    radius = np.random.randint(30, 80)
    dist = np.sqrt((x-center_x)**2 + (y-center_y)**2)
    mask = dist < radius
    base[mask] = (base[mask] * 0.6).astype(np.uint8)
    noise = np.random.randint(-20, 20, (size, size, 3))
    img = np.clip(base.astype(int) + noise, 0, 255).astype(np.uint8)
    Image.fromarray(img).save(path)

os.makedirs('data/labeled', exist_ok=True)

print(f"Creating {len(all_images)} synthetic labeled images...")
for img_id in tqdm(all_images):
    path = f'data/labeled/{img_id}.jpg'
    if not os.path.exists(path):
        create_synthetic_skin_image(path)

print(f"Created {len(os.listdir('data/labeled'))} images")

Creating 691 synthetic labeled images...


100%|██████████| 691/691 [00:04<00:00, 169.27it/s]

Created 691 images


#**Training with Rebalancing**

In [34]:
class_counts = train_df['label'].value_counts()
print("Class counts:", class_counts.to_dict())

# Make MEL weight 10x higher than NV
weights = torch.tensor([
    10.0,  # MEL (rarest, most important)
    5.0,   # BKL
    1.0    # NV (majority)
], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
print(f"Class weights: MEL={weights[0]:.1f}, BKL={weights[1]:.1f}, NV={weights[2]:.1f}")

# UNFREEZE ENCODER for better adaptation
for param in encoder.parameters():
    param.requires_grad = True

# Use different learning rates: lower for encoder, higher for classifier
optimizer = optim.Adam([
    {'params': encoder.parameters(), 'lr': 0.0001},  # Slower updates
    {'params': model.classifier.parameters(), 'lr': 0.001}  # Faster updates
])

Class counts: {'NV': 349, 'BKL': 99, 'MEL': 35}
Class weights: MEL=10.0, BKL=5.0, NV=1.0


#**Training without Early Stop applied**

In [36]:
print("\nFine-tuning with STRONG rebalancing (unfrozen encoder)...")
epochs = 20
best_val = 0

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_acc, _, _ = evaluate(model, val_loader)

    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), 'best_ssl_model_v2.pth')

         # Check if we're predicting minority classes now
    _, val_labels, val_preds = evaluate(model, val_loader)
    mel_preds = sum(1 for p in val_preds if p == 0)  # Class 0 is MEL

    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}: Train={train_acc:.3f}, Val={val_acc:.3f}, MEL predictions: {mel_preds}")

print(f"\nBest val accuracy: {best_val:.3f}")


Fine-tuning with STRONG rebalancing (unfrozen encoder)...
Epoch 2: Train=0.205, Val=0.202, MEL predictions: 104
Epoch 4: Train=0.205, Val=0.202, MEL predictions: 104
Epoch 6: Train=0.205, Val=0.202, MEL predictions: 104
Epoch 8: Train=0.205, Val=0.202, MEL predictions: 104
Epoch 10: Train=0.205, Val=0.202, MEL predictions: 104
Epoch 12: Train=0.205, Val=0.202, MEL predictions: 104
Epoch 14: Train=0.205, Val=0.202, MEL predictions: 104
Epoch 16: Train=0.205, Val=0.202, MEL predictions: 104
Epoch 18: Train=0.205, Val=0.202, MEL predictions: 104
Epoch 20: Train=0.205, Val=0.202, MEL predictions: 104

Best val accuracy: 0.202


#**Final Evaluation**

In [37]:
model.load_state_dict(torch.load('best_ssl_model_v2.pth'))
test_acc, true_labels, pred_labels = evaluate(model, test_loader)

# Check predictions distribution
from collections import Counter
pred_dist = Counter(pred_labels)
print(f"\nPrediction distribution: {pred_dist}")

print("\nPer-Class Performance:")
print(classification_report(true_labels, pred_labels, target_names=train_ds.classes))

mel_recall = report[train_ds.classes[0]]['recall']
print(f"\n>>> MEL Recall: {mel_recall:.1%} <<<")


Prediction distribution: Counter({np.int64(0): 104})

Per-Class Performance:
              precision    recall  f1-score   support

         BKL       0.20      1.00      0.34        21
         MEL       0.00      0.00      0.00         8
          NV       0.00      0.00      0.00        75

    accuracy                           0.20       104
   macro avg       0.07      0.33      0.11       104
weighted avg       0.04      0.20      0.07       104


>>> MEL Recall: 0.0% <<<


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#**Backing up current findings**

In [38]:
# Backup current results
import json
with open('current_status.json', 'w') as f:
    json.dump({
        'status': 'Phase 1 - Pipeline complete, tuning in progress',
        'baseline_mel_recall': 0.0,
        'ssl_encoder_trained': True,
        'fine_tuning_status': 'Architecture ready, class weights need calibration',
        'next_steps': ['Grid search class weights', 'Try focal loss', 'Longer training']
    }, f, indent=2)

# Save to Drive
import shutil
shutil.copy('current_status.json', '/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1/')
print("Status saved. Ready for defense.")

Status saved. Ready for defense.
